# 0. Install and Import Dependencies

In [2]:
!pip install mediapipe opencv-python


[notice] A new release of pip is available: 23.2.1 -> 24.1.1
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip


In [1]:
import cv2
import mediapipe as mp
import numpy as np
import requests 
import time
import threading
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# 1. Determining Joints

<img src="https://i.imgur.com/3j8BPdc.png" style="height:300px" >

# 2. Calculate Angles

In [3]:
def calculate_angle(a,b,c):
    a = np.array(a) # First
    b = np.array(b) # Mid
    c = np.array(c) # End
    
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)
    
    if angle >180.0:
        angle = 360-angle
        
    return angle 

# 3. Note Determination

In [4]:
 
# Define a lookup table for note determination
NOTE_LOOKUP_TABLE = {
    # angle elbow, angle shoulder
    ((135, 180), (  0,  45)): "1",
    ((135, 180), ( 45,  90)): "2",
    (( 90, 135), (  0, 180)): "3",
    (( 45,  90), (  0, 180)): "4",
    ((  0,  45), (  0, 180)): "5",
}

# Determine the note based on the angles using a lookup table
def determine_note(angle_elbow, angle_shoulder):
    for ((elbow_min, elbow_max), (shoulder_min, shoulder_max)), note in NOTE_LOOKUP_TABLE.items():
        if elbow_min <= angle_elbow < elbow_max and shoulder_min <= angle_shoulder < shoulder_max:
            return note
    return "404"
 

In [5]:
def send_notes_to_server(url, noteManuale, noteTempo):
    try:
        requests.post(url, json={'noteManuale': noteManuale, 'noteTempo': noteTempo})
    except Exception as e:
        print(f"Error sending notes to server: {e}")

# 4. Main

In [6]:
# Main function to process video feed
def process_video():
    cap = cv2.VideoCapture(0)
    url = 'http://127.0.0.1:5000/send_value'

    last_sent_time = 0
    with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            # Recolor image to RGB
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False

            # Make detection
            results = pose.process(image)

            # Recolor back to BGR
            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            # Extract landmarks
            try:
                landmarks = results.pose_landmarks.landmark

                # Get coordinates
                hipLeft       = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x,landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
                hipRight      = [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x,landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y]
                shoulderLeft  = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
                shoulderRight = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x,landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
                elbowLeft     = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x,landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
                elbowRight    = [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x,landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y]
                wristLeft     = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x,landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]
                wristRight    = [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x,landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y]

                # Calculate angle
                angleElbowLeft     = calculate_angle(shoulderLeft, elbowLeft, wristLeft)
                angleElbowRight    = calculate_angle(shoulderRight, elbowRight, wristRight)
                angleShoulderLeft  = calculate_angle(hipLeft, shoulderLeft, elbowLeft)
                angleShoulderRight = calculate_angle(hipRight, shoulderRight, elbowRight)

                # Determine notes
                noteManuale = determine_note(angleElbowRight, angleShoulderRight)
                noteTempo   = determine_note(angleElbowLeft , angleShoulderLeft )
                current_time = time.time()
                if current_time - last_sent_time >= 0.25:
                    threading.Thread(target=send_notes_to_server, args=(url, noteManuale, noteTempo)).start()
                    last_sent_time = current_time
                    
            except Exception as e:
                print(f"Error extracting landmarks or calculating angles: {e}")
                noteManuale, noteTempo = "NoN", "NoN"

            # Display note
            display_note_box(image, noteManuale, noteTempo)

            # Render detections
            if results.pose_landmarks:
                mp_drawing.draw_landmarks(
                    image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                    mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=2),
                    mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2)
                )

            cv2.imshow('Mediapipe Feed', image)

            if cv2.waitKey(10) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
 

# 4. Display Note

In [7]:
# Helper function to display the note
def display_note_box(image, noteManuale, noteTempo):
    # Draw the box
    cv2.rectangle(image, (0, 0), (300, 73), (245, 117, 16), -1)

    # Display the text 'Note Manuale' and 'Note Tempo'
    cv2.putText(image, 'Manuale:', (10, 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)
    cv2.putText(image, noteManuale, (150, 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)
    
    cv2.putText(image, 'Tempo:', (10, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)
    cv2.putText(image, noteTempo, (150, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)

In [8]:
process_video()

Error extracting landmarks or calculating angles: 'NoneType' object has no attribute 'landmark'
Error extracting landmarks or calculating angles: 'NoneType' object has no attribute 'landmark'
Error extracting landmarks or calculating angles: 'NoneType' object has no attribute 'landmark'
Error extracting landmarks or calculating angles: 'NoneType' object has no attribute 'landmark'
Error sending notes to server: HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /send_value (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000001E91E545660>: Failed to establish a new connection: [WinError 10061] Подключение не установлено, т.к. конечный компьютер отверг запрос на подключение'))
Error sending notes to server: HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /send_value (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000001E91E500DC0>: Failed to establish a new connection:

KeyboardInterrupt: 